In [ ]:
import sys
sys.path.append('/kaggle/input/amp-pd')

In [ ]:
import numpy as np
import pandas as pd

# 项目 visit_history 模型的 time 基线：12 个月核加权，最小化经验 SMAPE+1。
data_dir = '/kaggle/input/amp-pd'
clinical = pd.read_csv(data_dir + '/train_clinical_data.csv')
keys = ['visit_id', 'patient_id', 'visit_month']
origins = pd.concat([
    pd.read_csv(data_dir + '/train_proteins.csv', usecols=keys),
    pd.read_csv(data_dir + '/train_peptides.csv', usecols=keys),
]).drop_duplicates()
targets = ['updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']
labels = clinical.melt(id_vars=['patient_id', 'visit_month'], value_vars=targets,
                       var_name='target', value_name='rating').dropna(subset=['rating'])
labels = labels.rename(columns={'visit_month': 'target_month'})
requests = origins.merge(pd.DataFrame({'horizon': [0, 6, 12, 24]}), how='cross')
requests['target_month'] = requests['visit_month'] + requests['horizon']
train = requests.merge(labels, on=['patient_id', 'target_month'], how='inner')

def predict(sample_submission):
    parts = sample_submission['prediction_id'].str.extract(
        r'^.+_(?P<visit_month>\d+)_(?P<target>updrs_[1-4])_plus_(?P<horizon>0|6|12|24)_months$'
    )
    parts['target_month'] = parts['visit_month'].astype(int) + parts['horizon'].astype(int)
    result = sample_submission.copy()
    for (target, month), rows in parts.groupby(['target', 'target_month']):
        selected = train[train['target'] == target]
        truth = selected['rating'].to_numpy()[:, None]
        candidates = np.arange(int(truth.max()) + 1)
        losses = 200 * np.abs(truth - candidates) / (truth + candidates + 2)
        weights = np.exp(-0.5 * ((selected['target_month'].to_numpy() - month) / 12) ** 2)
        result.loc[rows.index, 'rating'] = candidates[(weights @ losses).argmin()]
    return result


In [ ]:
import amp_pd_peptide
env = amp_pd_peptide.make_env()
iter_test = env.iter_test()

In [ ]:
# 与官方 demo 相同：每批交回该批模板。
for (test, test_peptides, test_proteins, sample_submission) in iter_test:
    sample_submission = predict(sample_submission)
    env.predict(sample_submission)
